# 01. Ollama の stop シーケンス検証 ← **最重要**

`biomni/llm.py` の Ollama 分岐は `stop_sequences` を渡していない。

```python
elif source == "Ollama":
    return ChatOllama(model=model, temperature=temperature)   # ← stop が無い
```

A1 は `</execute>` で生成を止め、コードを実行して `<observation>` を差し込むことで
ReAct ループを成立させている。stop が効かないと、**モデルが実行していないコードの
「実行結果」を自分で書く**。根拠提示アプリではこれが最悪の故障モードなので、
最初にここを潰す（受け入れ基準 AC-1）。

このノートブックは Ollama が必要。

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
from biomni_hypo.config import Settings, apply_biomni_env, install_hint, missing_dependencies
from biomni_hypo.llm import (
    AGENT_STOP_SEQUENCES,
    build_chat_ollama,
    hallucinated_observation,
    ollama_status,
)

missing = missing_dependencies()
assert not missing, f"依存が足りません。実行してください: {install_hint(missing)}"

settings = Settings()
apply_biomni_env(settings)
assert ollama_status(settings.ollama_base_url).reachable, "Ollama に到達できません（00 を先に）"
print("stop シーケンス:", AGENT_STOP_SEQUENCES)


## A1 のフォーマットを再現した最小プロンプト

A1 の巨大なシステムプロンプトは使わず、`<execute>` を書かせる最小の指示だけを与える。
これで「stop が効いているか」だけを切り出して見る。

In [ ]:
SYSTEM = """You are a research agent. To run code, wrap it in <execute> tags:
<execute>
print(1 + 1)
</execute>
After </execute> the system will run the code and give you the result in <observation> tags.
Never write <observation> yourself."""

TASK = "Compute 2 + 2 in Python and tell me the answer."

def ask(llm):
    try:
        from langchain_core.messages import HumanMessage, SystemMessage
        msgs = [SystemMessage(content=SYSTEM), HumanMessage(content=TASK)]
    except ImportError:
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": TASK}]
    return llm.invoke(msgs).content

## A. biomni の `get_llm()` をそのまま使った場合（stop なし）

In [ ]:
from biomni.llm import get_llm

llm_biomni = get_llm(settings.model, source="Ollama", stop_sequences=AGENT_STOP_SEQUENCES)
raw_biomni = ask(llm_biomni)

print(raw_biomni[:1200])
print("\n" + "=" * 60)
print("observation を自己生成した:", hallucinated_observation(raw_biomni))

## B. 本アプリの `build_chat_ollama()`（stop あり）

In [ ]:
llm_ours = build_chat_ollama(settings, stop=AGENT_STOP_SEQUENCES)
raw_ours = ask(llm_ours)

print(raw_ours[:1200])
print("\n" + "=" * 60)
print("observation を自己生成した:", hallucinated_observation(raw_ours))
print("</execute> で止まった      :", raw_ours.rstrip().endswith("</execute>") or "</execute>" not in raw_ours)

## 判定

- **B が `False`（自己生成なし）**: 対策が効いている。02 へ進む。
- **B が `True`**: `ChatOllama` に `stop` が渡っていない。`langchain-ollama` のバージョンを確認
  （古い版は `stop_sequences` という引数名）。`build_chat_ollama()` の 1 箇所を直せば全体に効く。
- **A も `False`**: そのモデルがたまたま `<observation>` を書かなかっただけの可能性がある。
  下のセルで複数回試すこと。

In [ ]:
# 複数回試して安定性を見る（1 回の成功は偶然かもしれない）
N = 3
for label, llm in [("stopなし(biomni)", llm_biomni), ("stopあり(本アプリ)", llm_ours)]:
    hits = sum(hallucinated_observation(ask(llm)) for _ in range(N))
    print(f"{label:22s}: {N} 回中 {hits} 回で observation を自己生成")

## num_ctx も確認する

Ollama の既定 context は 2048。A1 のシステムプロンプト（ツール説明を含む）だけで軽く超える。
`build_chat_ollama()` は `settings.num_ctx` を渡している。

In [ ]:
print("num_ctx  :", getattr(llm_ours, "num_ctx", None))
print("stop     :", getattr(llm_ours, "stop", None))
print("base_url :", getattr(llm_ours, "base_url", None))
print()
print("※ biomni の get_llm() は base_url を Ollama に渡さない（§4.2）:")
print("   biomni 版 base_url:", getattr(llm_biomni, "base_url", None))

---

## Ollama が無い場合: モックで「配線」だけ検証する

`biomni_hypo/mock_ollama.py` は Ollama の HTTP API を最小限だけ実装したテスト用サーバ。
**設定が実際にリクエストへ乗っているか**は、実機が無くてもここで確かめられる。

- 確かめられる: `stop` / `num_ctx` / `base_url` が送信されているか
- 確かめられない: モデルが本当に `</execute>` で止まるか（それは上のセルで）

In [ ]:
from biomni_hypo.mock_ollama import MockOllama
from biomni_hypo.llm import build_chat_ollama

with MockOllama(replies=["mock からの応答"]) as mock:
    s = Settings()
    s.ollama_base_url = mock.base_url
    s.num_ctx = 12345

    # A. biomni の get_llm（base_url を無視するので OLLAMA_HOST でしか向けられない）
    import os
    os.environ["OLLAMA_HOST"] = mock.base_url
    from biomni.llm import get_llm
    get_llm("qwen3:14b", source="Ollama", stop_sequences=AGENT_STOP_SEQUENCES,
            base_url=mock.base_url).invoke([{"role": "user", "content": "hi"}])
    print("A. biomni get_llm   ->", mock.last_options())

    # B. 本アプリ
    build_chat_ollama(s, stop=AGENT_STOP_SEQUENCES).invoke([{"role": "user", "content": "hi"}])
    print("B. build_chat_ollama ->", mock.last_options())

A に `stop` と `num_ctx` が無く、B にあれば対策が効いている。
同じことを `tests/test_integration_biomni.py` が自動テストとして固定している。

```bash
pytest tests/test_integration_biomni.py -q
```